# Elevation and environmental features with XGBoost
Compare the same XGBoost settings and baseline columns with/without `elevation_m`,
and (when present) the combined FEMA/CAL FIRE features from the earlier Random Forest.
Toy-data split: January 2024 training, February 2024 validation, March 2024 test.
The selected variant is refitted on January + February before testing on March.
No charts, tuning, or permutation analysis.
Edit the CSV path and months below. The CSV must already contain acquired risk features.
Models predict `ClosePrice` directly; MAE/RMSE are dollars and MAPE is percent.
Compare variants on validation first. Run the final cell once after choosing a variant.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor  # If missing, run %pip install xgboost in this kernel.

DATA_PATH = Path("train_toy_risk.csv")  # January–March 2024 toy data with risk features.
RESULTS_DIR = Path("results")
VALIDATION_MONTH = "2024-02"
TEST_MONTH = "2024-03"
TRAINING_WINDOW = 1  # January training; final fit includes January + February.
TARGET = "ClosePrice"

# Use the same tuned parameters for every feature variant. No search is performed here.
XGB_PARAMS = dict(
    objective="reg:squarederror", tree_method="hist", n_estimators=600,
    max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42,
)

In [6]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set DATA_PATH to your risk-enriched CSV: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH, dtype={"PostalCode": "string"})
required = {TARGET, "CloseDate", "elevation_m"}
if missing := required.difference(df.columns):
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce", utc=True).dt.tz_convert(None)
valid = np.isfinite(df[TARGET]) & df[TARGET].gt(0) & df["CloseDate"].notna()
print(f"Discarding {(~valid).sum():,} rows with invalid target/date.")
df = df.loc[valid].copy()
df["sale_year"] = df["CloseDate"].dt.year
df["sale_month"] = df["CloseDate"].dt.month

BASE_NUMERIC = [
    "AttachedGarageYN", "BathroomsTotalInteger", "BedroomsTotal", "FireplaceYN",
    "GarageSpaces", "Latitude", "Longitude", "LivingArea", "LotSizeSquareFeet",
    "NewConstructionYN", "ParkingTotal", "PoolPrivateYN", "Stories", "ViewYN",
    "YearBuilt", "sale_year", "sale_month",
]
BASE_CATEGORICAL = ["City", "CountyOrParish", "Flooring", "Levels", "PostalCode", "HighSchoolDistrict"]
RISK_NUMERIC = ["fema_sfha"]
RISK_CATEGORICAL = ["fema_fld_zone", "fema_zone_subty", "fire_lra_fhsz_description", "fire_sra_fhsz_description"]
for col in BASE_NUMERIC + RISK_NUMERIC + ["elevation_m"]:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
# USGS no-data sentinel; negative real elevations are valid.
df.loc[df["elevation_m"].le(-100000), "elevation_m"] = np.nan
for col in BASE_CATEGORICAL + RISK_CATEGORICAL:
    if col in df:
        df[col] = df[col].astype("string").str.strip().replace("", pd.NA)
        df[col] = df[col].astype(object).where(df[col].notna(), np.nan)

val_start = pd.Period(VALIDATION_MONTH, freq="M").start_time
test_start = pd.Period(TEST_MONTH, freq="M").start_time
if not 0 < TRAINING_WINDOW or test_start <= val_start:
    raise ValueError("Use a positive training window and TEST_MONTH later than VALIDATION_MONTH.")
train_start = val_start - pd.DateOffset(months=TRAINING_WINDOW)
train = df.loc[df["CloseDate"].ge(train_start) & df["CloseDate"].lt(val_start)].copy()
validation = df.loc[df["CloseDate"].ge(val_start) & df["CloseDate"].lt(val_start + pd.DateOffset(months=1))].copy()
test = df.loc[df["CloseDate"].ge(test_start) & df["CloseDate"].lt(test_start + pd.DateOffset(months=1))].copy()
for name, part in [("Training", train), ("Validation", validation), ("Test", test)]:
    if len(part) < 2:
        raise ValueError(f"{name} has {len(part)} rows. Change the configured months or supply more data.")
    print(f"{name}: {len(part):,} rows; {part['CloseDate'].min().date()} to {part['CloseDate'].max().date()}")

Discarding 0 rows with invalid target/date.
Training: 8,305 rows; 2024-01-01 to 2024-01-31
Validation: 9,544 rows; 2024-02-01 to 2024-02-29
Test: 6,569 rows; 2024-03-01 to 2024-03-31


In [7]:
# Select usable columns using training data only; preserve identical rows for every model.
def usable(columns):
    return [c for c in columns if c in train and train[c].notna().any()]

baseline = usable(BASE_NUMERIC + BASE_CATEGORICAL)
risk = usable(RISK_NUMERIC + RISK_CATEGORICAL)
if not baseline:
    raise ValueError("No usable baseline features found.")
if train["elevation_m"].notna().sum() < 2 or train["elevation_m"].nunique() < 2:
    raise ValueError("Training elevation needs at least two observed, distinct values.")
VARIANTS = {
    "Baseline": baseline,
    "Baseline + elevation": baseline + ["elevation_m"],
}
if risk:
    VARIANTS["Combined without elevation"] = baseline + risk
    VARIANTS["Combined + elevation"] = baseline + risk + ["elevation_m"]
else:
    print("No usable flood/fire columns; comparing only baseline vs elevation.")
print("Baseline columns:", baseline)
print("Additional flood/fire columns:", risk)
print("Elevation coverage:", {name: f"{part['elevation_m'].notna().mean():.1%}" for name, part in [("train", train), ("validation", validation), ("test", test)]})

CATEGORICAL = set(BASE_CATEGORICAL + RISK_CATEGORICAL)
def make_model(features):
    numeric = [c for c in features if c not in CATEGORICAL]
    categorical = [c for c in features if c in CATEGORICAL]
    transforms = []
    if numeric:
        transforms.append(("numeric", SimpleImputer(strategy="median"), numeric))
    if categorical:
        transforms.append(("categorical", Pipeline([
            ("missing", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore", dtype=np.float32)),
        ]), categorical))
    return Pipeline([
        ("preprocess", ColumnTransformer(transforms, sparse_threshold=1.0)),
        ("xgboost", XGBRegressor(**XGB_PARAMS)),
    ])

def score(model, part, features):
    actual = part[TARGET].to_numpy()
    predicted = model.predict(part.loc[:, features])
    return dict(mae=mean_absolute_error(actual, predicted),
                rmse=mean_squared_error(actual, predicted) ** 0.5,
                r2=r2_score(actual, predicted),
                mape_pct=100 * mean_absolute_percentage_error(actual, predicted))

Baseline columns: ['AttachedGarageYN', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN', 'GarageSpaces', 'Latitude', 'Longitude', 'LivingArea', 'LotSizeSquareFeet', 'NewConstructionYN', 'ParkingTotal', 'PoolPrivateYN', 'ViewYN', 'sale_year', 'sale_month', 'City', 'CountyOrParish', 'PostalCode']
Additional flood/fire columns: ['fema_sfha', 'fema_fld_zone', 'fema_zone_subty', 'fire_lra_fhsz_description', 'fire_sra_fhsz_description']
Elevation coverage: {'train': '100.0%', 'validation': '74.8%', 'test': '100.0%'}


In [8]:
rows = []
for name, features in VARIANTS.items():
    print(f"Fitting {name} ({len(features)} input columns)...", flush=True)
    model = make_model(features)
    model.fit(train.loc[:, features], train[TARGET].to_numpy())
    rows.append(dict(variant=name, feature_count=len(features), **score(model, validation, features)))
results = pd.DataFrame(rows)
base_mae = results.loc[results["variant"].eq("Baseline"), "mae"].iloc[0]
results["mae_change_vs_baseline"] = results["mae"] - base_mae
results = results.sort_values("mae").reset_index(drop=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_DIR / "elevation_xgboost_validation.csv", index=False)
display(results.round(4))  # Negative MAE change means improvement.

Fitting Baseline (18 input columns)...
Fitting Baseline + elevation (19 input columns)...
Fitting Combined without elevation (23 input columns)...
Fitting Combined + elevation (24 input columns)...


,variant,feature_count,mae,rmse,r2,mape_pct,mae_change_vs_baseline
0,Baseline,18,281397.6089,798081.0172,0.6404,20.3903,0.0000
1,Baseline + elevation,19,281674.0767,788978.9298,0.6486,20.3405,276.4678
2,Combined + elevation,24,284280.5555,797363.4080,0.6410,20.3667,2882.9466
3,Combined without elevation,23,287404.8371,800612.7055,0.6381,20.5054,6007.2282


In [13]:
development = df.loc[
    df["CloseDate"].ge(train_start)
    & df["CloseDate"].lt(test_start)
].copy()

test_rows = []

for i in range(4):
    SELECTED_VARIANT = results.iloc[i]["variant"]
    features = VARIANTS[SELECTED_VARIANT]

    final_model = make_model(features)
    final_model.fit(
        development.loc[:, features],
        development[TARGET].to_numpy(),
    )

    test_rows.append(
        dict(
            variant=SELECTED_VARIANT,
            training_rows=len(development),
            test_rows=len(test),
            **score(final_model, test, features),
        )
    )

test_results = pd.DataFrame(test_rows)
display(test_results.round(4))

,variant,training_rows,test_rows,mae,rmse,r2,mape_pct
0,Baseline,17849,6569,198961.9096,478301.2545,0.8334,14.0578
1,Baseline + elevation,17849,6569,196772.7520,463655.6035,0.8435,14.0465
2,Combined + elevation,17849,6569,196659.3965,475008.2011,0.8357,13.9292
3,Combined without elevation,17849,6569,194910.4121,462786.6039,0.8440,13.9498


In [14]:
test_results.to_csv(RESULTS_DIR / "elevation_xgboost_test.csv", index=False)